In [ ]:
import pandas as pd

In [ ]:
X_train = pd.read_csv(r'/kaggle/input/datasets/masterzero1/higgs-dat/X_train.csv')
X_val   = pd.read_csv(r'/kaggle/input/datasets/masterzero1/higgs-dat/X_val.csv')
X_test  = pd.read_csv(r'/kaggle/input/datasets/masterzero1/higgs-dat/X_test.csv')
y_train = pd.read_csv(r'/kaggle/input/datasets/masterzero1/higgs-dat/y_train.csv').squeeze()
y_val   = pd.read_csv(r'/kaggle/input/datasets/masterzero1/higgs-dat/y_val.csv').squeeze()
y_test  = pd.read_csv(r'/kaggle/input/datasets/masterzero1/higgs-dat/y_test.csv').squeeze()

In [ ]:
#scalling the features columns (for logistic regression):

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
low_lvl_features = ['lepton  pT', 'lepton  eta', 'lepton  phi', 'missing energy magnitude', 'missing energy phi', 'jet 1 pt', 'jet 1 eta', 'jet 1 phi', 'jet 1 b-tag', 'jet 2 pt', 'jet 2 eta', 'jet 2 phi', 'jet 2 b-tag', 'jet 3 pt', 'jet 3 eta', 'jet 3 phi', 'jet 3 b-tag', 'jet 4 pt', 'jet 4 eta', 'jet 4 phi', 'jet 4 b-tag']
high_lvl_features = ['m_jj', 'm_jjj', 'm_lv', 'm_jlv', 'm_bb', 'm_wbb', 'm_wwbb']

#for logistic regression

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

X_train_28_scaled = X_train_scaled
X_train_21_scaled = X_train_scaled.drop(high_lvl_features, axis=1)
X_train_7_scaled = X_train_scaled.drop(low_lvl_features, axis=1)

X_val_28_scaled = X_val_scaled
X_val_21_scaled = X_val_scaled.drop(high_lvl_features, axis=1)
X_val_7_scaled = X_val_scaled.drop(low_lvl_features, axis=1)

X_test_28_scaled = X_test_scaled
X_test_21_scaled = X_test_scaled.drop(high_lvl_features, axis=1)
X_test_7_scaled = X_test_scaled.drop(low_lvl_features, axis=1)

#for xgboost
X_train_28 = X_train
X_train_21 = X_train.drop(high_lvl_features, axis=1)
X_train_7 = X_train.drop(low_lvl_features, axis=1)

X_val_28 = X_val
X_val_21 = X_val.drop(high_lvl_features, axis=1)
X_val_7 = X_val.drop(low_lvl_features, axis=1)

X_test_28 = X_test
X_test_21 = X_test.drop(high_lvl_features, axis=1)
X_test_7 = X_test.drop(low_lvl_features, axis=1)

In [ ]:
#Logistic Regression (baseline model):

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

#logistic regression on 21 low level features
log_reg_21 = LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=42)
log_reg_21.fit(X_train_21_scaled, y_train)
auc_21 = roc_auc_score(y_val, log_reg_21.predict_proba(X_val_21_scaled)[:, 1])
print(f"Logistic Regression | 21 features | AUC: {auc_21:.4f}")

#logistic regression on 7 high level features
log_reg_7 = LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=42)
log_reg_7.fit(X_train_7_scaled, y_train)
auc_7 = roc_auc_score(y_val, log_reg_7.predict_proba(X_val_7_scaled)[:, 1])
print(f"Logistic Regression | 7 features | AUC: {auc_7:.4f}")

#logistic regression on all 28 features
log_reg_28 = LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=42)
log_reg_28.fit(X_train_28_scaled, y_train)
auc_28 = roc_auc_score(y_val, log_reg_28.predict_proba(X_val_28_scaled)[:, 1])
print(f"Logistic Regression | 28 features | AUC: {auc_28:.4f}")

Logistic Regression | 21 features | AUC: 0.5944    
Logistic Regression | 7 features | AUC: 0.6451    
Logistic Regression | 28 features | AUC: 0.6831    

As we can see the results of the logistic regression are not so good, for eg on the 21 low level features, the auc score is 0.59 which is barely better than random guessing

In [ ]:
#XGBoost (XGBClassifier)

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

#xgboost for 21 low level features
xgb_21 = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    device='cuda'
)

xgb_21.fit(X_train_21, y_train)
auc_21 = roc_auc_score(y_val, xgb_21.predict_proba(X_val_21)[:, 1])
print(f"XGBoost Default | 21 features | AUC: {auc_21:.4f}")

#xgboost for 7 high level features
xgb_7 = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    device='cuda'
)

xgb_7.fit(X_train_7, y_train)
auc_7 = roc_auc_score(y_val, xgb_7.predict_proba(X_val_7)[:, 1])
print(f"XGBoost Default | 7 features | AUC: {auc_7:.4f}")

#xgboost for all 28 features
xgb_28 = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    device='cuda'
)

xgb_28.fit(X_train_28, y_train)
auc_28 = roc_auc_score(y_val, xgb_28.predict_proba(X_val_28)[:, 1])
print(f"XGBoost Default | 28 features | AUC: {auc_28:.4f}")



The results for the XGBoost model are really good, so comparing it to the P. Baldi research paper it already beats the TMVA BDT and it beats the Shallow NN too

XGBoost Default | 21 features | AUC: 0.7265    
XGBoost Default | 7 features | AUC: 0.7899    
XGBoost Default | 28 features | AUC: 0.8240    

In [ ]:
#running the hyperparameter tuning using RandomizedSearchCV

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score

param_dist = {
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5, 7],
    'n_estimators': [100, 200, 300, 400, 500],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.5, 1, 1.5, 2],
}

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    device='cuda',
    early_stopping_rounds=20
)

random_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=50,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

random_search.fit(
    X_train_28, y_train,
    eval_set=[(X_val_28, y_val)],
    verbose=False
)

print("Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

print(f"Best CV AUC: {random_search.best_score_:.4f}")

auc_tuned = roc_auc_score(
    y_val,
    random_search.best_estimator_.predict_proba(X_val_28)[:, 1]
)
print(f"Validation AUC: {auc_tuned:.4f}")

Parameters found from RandomizedSearchCV

Best params:  
  subsample: 0.9    
  reg_lambda: 1   
  reg_alpha: 0.1    
  n_estimators: 500   
  min_child_weight: 3   
  max_depth: 8    
  learning_rate: 0.1    
  gamma: 0    
  colsample_bytree: 0.7   
  
Best CV AUC: 0.8388   
Validation AUC: 0.8390  

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score

param_dist = {
    'max_depth': [8, 9, 10, 11, 12],
    'learning_rate': [0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5, 7],
    'n_estimators': [500, 600, 700, 800, 900, 1000],
    'gamma': [0, 0.01, 0.05, 0.1, 0.2],
    'reg_alpha': [0.01, 0.05, 0.1, 0.2, 0.5],
    'reg_lambda': [0.5, 1, 1.5, 2]
}

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    device='cuda'
)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=100,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train_28, y_train)

print("Best params:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

print(f"Best CV AUC: {random_search.best_score_:.4f}")

auc_tuned = roc_auc_score(
    y_val,
    random_search.best_estimator_.predict_proba(X_val_28)[:, 1]
)

print(f"Validation AUC: {auc_tuned:.4f}")